# 02. Data Cleaning & Text Preprocessing

This notebook handles data sanitization, null imputation, duplicate removal, text normalization, lowercasing, special character stripping, tokenization, and stopword removal.

### Why Preprocessing is Essential in NLP:
- **Lowercasing**: Prevents treating 'Great' and 'great' as distinct features.
- **Regex Clean**: Removes HTML tags, URLs, and noisy punctuation.
- **Stopword Removal**: Eliminates non-informative words like 'is', 'the', 'at'.


In [5]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))


## Step 1: Load Raw Dataset


In [6]:
df = pd.read_csv('../raw/product_reviews.csv')
print('Initial row count:', len(df))
df.head(3)


Initial row count: 10000


,product_id,product_name,category,review_title,review_text,rating,review_date,verified_purchase
0,AMZ-B000PHHY8O,Amazon Grocery Item (B000PHHY),Grocery & Gourmet,Retro Candy Gift Box,I bought this at Christmas time for my much ol...,5,2012-05-17,True
1,AMZ-B0029NTQ1K,Amazon Grocery Item (B0029NTQ),Grocery & Gourmet,Your cats will crave it fortnightly!,"Actually, if your cats are like mine, they wil...",5,2010-12-16,True
2,AMZ-B001EPPSYI,Amazon Grocery Item (B001EPPS),Grocery & Gourmet,Wonderful treat,This tea is like sipping a mint flavored choco...,5,2010-03-15,True


## Step 2: Combine Title and Body Text


In [7]:
df['full_review'] = df['review_title'].fillna('') + ' ' + df['review_text'].fillna('')
df['full_review'].head(3)


0    Retro Candy Gift Box I bought this at Christma...
1    Your cats will crave it fortnightly! Actually,...
2    Wonderful treat This tea is like sipping a min...
Name: full_review, dtype: object

## Step 3: Text Cleaning Function


In [8]:
def clean_review_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['full_review'].apply(clean_review_text)
df[['full_review', 'cleaned_text']].head(3)


,full_review,cleaned_text
0,Retro Candy Gift Box I bought this at Christma...,retro candy gift box i bought this at christma...
1,"Your cats will crave it fortnightly! Actually,...",your cats will crave it fortnightly actually i...
2,Wonderful treat This tea is like sipping a min...,wonderful treat this tea is like sipping a min...


## Step 4: Tokenization & Stopword Filtering


In [9]:
def tokenize_and_remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered = [word for word in tokens if word not in stop_words and len(word) > 2]
    return ' '.join(filtered)

df['processed_tokens'] = df['cleaned_text'].apply(tokenize_and_remove_stopwords)
df[['cleaned_text', 'processed_tokens']].head(3)


,cleaned_text,processed_tokens
0,retro candy gift box i bought this at christma...,retro candy gift box bought christmas time muc...
1,your cats will crave it fortnightly actually i...,cats crave fortnightly actually cats like mine...
2,wonderful treat this tea is like sipping a min...,wonderful treat tea like sipping mint flavored...


## Step 5: Save Cleaned Dataset to Processed Directory


In [10]:
processed_path = '../processed/cleaned_reviews.csv'
df.to_csv(processed_path, index=False)
print(f'[+] Successfully saved cleaned dataset to: {processed_path}')


[+] Successfully saved cleaned dataset to: ../processed/cleaned_reviews.csv


### Conclusion:
- All text data was normalized, tokenized, and stripped of uninformative noise.
- The cleaned output is saved as `data/processed/cleaned_reviews.csv` for vectorization and model training.
